# API MedGemma sur Colab (GPU) + tunnel ngrok

**Setup obligatoire :** `Exécution > Modifier le type d'exécution > T4 GPU`

Ce notebook réutilise le pipeline validé de `rsna_medgemma_inference_sans_fine-tuning_1.ipynb`
(description libre → classification par mots-clés → JSON construit en Python)
et l'expose en **API FastAPI identique à `api/main.py`**, joignable depuis le
frontend via une **URL publique ngrok**.

**Pourquoi :** le modèle (~17 Go en float32) fait crasher les PC sans GPU.
Ici il tourne en 4-bit (~5-6 Go) sur le T4 gratuit de Colab ; le frontend
n'envoie que l'image et reçoit le JSON.

```text
code.html (PC local) ──POST /predict──> URL ngrok ──tunnel──> FastAPI (Colab) ──> MedGemma 4-bit (T4)
```

> /!\\ Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.

In [ ]:
# Step 0 — reproductibilité (repris du notebook d'inférence de l'équipe)
import os, random

SEED = 42

def set_all_seeds(seed: int = SEED):
    """Fixe la seed partout (random/numpy/torch/CUDA) pour des runs reproductibles."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_all_seeds(SEED)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(f'Seed fixée à {SEED}')

In [ ]:
# Step 1 — dépendances (une seule fois)
# fastapi/uvicorn : le serveur API | pyngrok : le tunnel public | python-multipart : upload fichiers
%pip install -q "transformers>=4.51.3" accelerate bitsandbytes pillow pandas fastapi "uvicorn[standard]" pyngrok python-multipart requests

In [ ]:
# Step 2 — clone du repo, pour réutiliser src/guardrails.py tel quel
# On ne recode pas les garde-fous : on importe ceux du repo,
# exactement comme le fait api/main.py en local.
import sys
from pathlib import Path

!git clone -q -b Medgemma-Test https://github.com/Vick-YE-Efrei/Solution-Delivery_DS-7B_ARVI-RX.git repo

PROJECT_ROOT = Path('/content/repo')
sys.path.append(str(PROJECT_ROOT))

from src.guardrails import apply_safety_guardrails, validate_prediction
print('Guardrails du repo importés.')

In [ ]:
# Step 3 — authentification HuggingFace (le modèle est en accès restreint)
# Conditions à accepter sur https://huggingface.co/google/medgemma-4b-pt
from huggingface_hub import login
login()

In [ ]:
# Step 4 — chargement de MedGemma-4B-PT (4-bit sur GPU)
# Repris du notebook d'inférence : 4-bit NF4 sur GPU, float32 sur CPU.
import warnings
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
warnings.filterwarnings('ignore')

MODEL_ID = 'google/medgemma-4b-pt'  # PT imposé

if torch.cuda.is_available():
    DEVICE, USE_4BIT = 'cuda', True
    print(f'GPU : {torch.cuda.get_device_name(0)}')
else:
    DEVICE, USE_4BIT = 'cpu', False
    print('CPU (lent, sans 4-bit) — pense à activer le GPU T4 !')

processor = AutoProcessor.from_pretrained(MODEL_ID)

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    # Quantization 4-bit (NF4) : réduit le modèle de ~17 Go (float32) à ~5-6 Go,
    # ce qui permet de le faire tourner sur le GPU T4 gratuit de Colab.
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map={'': 0}, dtype=torch.bfloat16)
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.float32, low_cpu_mem_usage=True)
    model.to(DEVICE)

model.eval()
print('Modèle chargé.  Device:', next(model.parameters()).device)

In [ ]:
# Step 5 — description libre -> classification par mots-clés -> JSON
# (même pipeline que rsna_medgemma_inference_sans_fine-tuning_1.ipynb)
import json, re, time
from PIL import Image

BOI = processor.tokenizer.boi_token  # token image

def build_prompt():
    return (
        f'{BOI}\n'
        'You are reading a frontal chest X-ray.\n'
        'Describe the lung fields and state whether there is any opacity, '
        'consolidation, or whether the lungs are clear.\n'
        'Findings:'
    )

WARNING_TEXT = ('Educational prototype only. Not for diagnosis. '
                'A qualified clinician must verify the image.')

OPACITY_KW = ['opacity', 'opacities', 'opacified', 'opacification',
              'consolidation', 'consolidations', 'pneumonia', 'infiltrate',
              'infiltration', 'airspace', 'air space', 'effusion', 'effusions',
              'haziness', 'hazy', 'nodule', 'nodules', 'mass', 'masses',
              'atelectasis', 'density', 'densities', 'reticular']
NORMAL_KW = ['normal', 'clear', 'unremarkable', 'no acute', 'no focal',
             'within normal limits', 'no abnormal', 'no significant']
CLEAR_KW = ['clear']
HEDGE_KW = ['possible', 'possibly', 'may represent', 'cannot exclude',
            'cannot be excluded', 'difficult to assess', 'questionable',
            'equivocal', 'suboptimal', 'underpenetrated', 'poorly',
            'low lung volumes', 'suspicious', 'concerning for']
NEG_TERMS = ['no', 'without', 'free of', 'negative for', 'absence of']

def _positive_hit(text, keywords):
    """Cherche un mot-clé NON nié (ignore 'no opacity', 'without consolidation')."""
    t = text.lower()
    for k in keywords:
        for m in re.finditer(re.escape(k), t):
            ctx = t[max(0, m.start() - 25):m.start()]
            if any(re.search(rf'\b{re.escape(n)}\b', ctx) for n in NEG_TERMS):
                continue
            return k
    return None

def classify_from_text(text):
    """Retourne (classe, confiance heuristique, evidence)."""
    op = _positive_hit(text, OPACITY_KW)
    clear = _positive_hit(text, CLEAR_KW)
    hedge = _positive_hit(text, HEDGE_KW)

    if op and clear:
        return 'uncertain', 0.50, [f"ambigu : '{op}' + 'clear'"]
    if op:
        return 'suspected_opacity', 0.70, [f"terme détecté : '{op}'"]
    if hedge:
        return 'uncertain', 0.50, [f"langage d'incertitude : '{hedge}'"]
    norm = _positive_hit(text, NORMAL_KW)
    if norm:
        return 'normal', 0.70, [f"terme détecté : '{norm}'"]
    return 'uncertain', 0.50, ['aucun terme discriminant clair']

def build_record(case_id, image_path, label, cls, conf, evidence, raw_text):
    """Assemble le dict final au format attendu par le frontend (mêmes clés que api/main.py)."""
    return {
        'case_id': case_id,
        'image_path': str(image_path),
        'label': label,
        'image_quality': 'good',
        'predicted_class': cls,
        'confidence': round(float(conf), 3),
        'visual_evidence': evidence + ([raw_text.strip()[:200]] if raw_text.strip() else []),
        'justification': ('Classe déduite par mots-clés depuis la description libre générée '
                          'par MedGemma-4b-pt. Résultat expérimental, non clinique.'),
        'limitations': [
            'modèle de base (pt) non instruction-tuned',
            'classification par mots-clés, non calibrée',
            'confiance heuristique, non probabiliste',
            'pas de contexte clinique',
        ],
        'warning': WARNING_TEXT,
    }

def predict(case_id, image_path, label=None, debug=False):
    """Génère une description de la radio avec MedGemma, puis la reclasse par mots-clés."""
    start = time.perf_counter()
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, text=build_prompt(),
                       return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False)

    gen = out[0][inputs['input_ids'].shape[1]:]  # ne garder que les tokens générés, pas le prompt
    raw_text = processor.decode(gen, skip_special_tokens=True)

    if debug:
        print('--- DESCRIPTION BRUTE GÉNÉRÉE ---')
        print(raw_text[:500])
        print('--- FIN ---')

    cls, conf, evidence = classify_from_text(raw_text)
    rec = build_record(case_id, image_path, label, cls, conf, evidence, raw_text)
    rec['latency_ms'] = int((time.perf_counter() - start) * 1000)
    rec['model_name'] = 'medgemma-4b-pt'
    return rec

print('predict() prête.')

In [ ]:
# Step 6 — API FastAPI, même structure que api/main.py du repo
import re as _re
import shutil
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title='Assistant radiologue virtuel EFREI — MedGemma sur Colab', version='0.1.0')

# CORS : autorise le frontend (code.html ouvert en local) à appeler cette API
app.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])

UPLOAD_DIR = Path('tmp_uploads')

@app.get('/')
def health():
    """Ping de santé, avec en plus le modèle et le device utilisés (utile pour déboguer sur Colab)."""
    return {'status': 'ok', 'scope': 'educational prototype, not diagnosis',
            'model': MODEL_ID, 'device': DEVICE}

@app.post('/predict')
async def predict_endpoint(file: UploadFile = File(...)):
    """Reçoit l'image du frontend, lance predict() (vrai MedGemma) et applique les garde-fous du repo."""
    # Sauvegarde de l'upload : logique identique à api/main.py
    UPLOAD_DIR.mkdir(exist_ok=True)
    filename = Path(file.filename or 'image.png').name
    suffix = Path(filename).suffix or '.png'
    stem = Path(filename).stem or 'image'
    safe_stem = _re.sub(r'[^A-Za-z0-9_.-]+', '_', stem)
    target = UPLOAD_DIR / f'uploaded_{safe_stem}{suffix}'
    with target.open('wb') as f:
        shutil.copyfileobj(file.file, f)

    # Vrai modèle au lieu de toy_predict, puis les MÊMES garde-fous que le repo
    pred = predict(safe_stem, target)
    return apply_safety_guardrails(pred)

print('API définie : GET / (health) et POST /predict')

In [ ]:
# Step 7 — lancement du serveur + tunnel ngrok, pour obtenir une URL publique
# Compte gratuit : https://dashboard.ngrok.com/signup
# Token :          https://dashboard.ngrok.com/get-started/your-authtoken
import threading
import uvicorn
from getpass import getpass
from pyngrok import ngrok

ngrok.set_auth_token(getpass('Colle ton token ngrok ici : '))

config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()

tunnel = ngrok.connect(8000)
API_URL = tunnel.public_url
print('API publique :', API_URL)
print('→ à coller dans le frontend (code.html)')
print('⚠ URL valable uniquement pendant cette session Colab')

In [ ]:
# Step 8 — auto-test : on appelle notre propre API comme le ferait le frontend,
# avec une image synthétique du repo
import requests

img = PROJECT_ROOT / 'data' / 'sample_images' / 'CXR_SYN_002_suspected_opacity.png'
with img.open('rb') as f:
    r = requests.post(f'{API_URL}/predict', files={'file': (img.name, f, 'image/png')})

print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## Brancher le frontend

1. Copier l'`API publique` affichée au Step 7 (ex. `https://a1b2-34-56.ngrok-free.app`)
2. La coller dans le champ « URL de l'API » du frontend (`code.html`)
3. Uploader une radiographie → l'image part vers Colab, le JSON revient

## Limites à connaître (et à assumer en soutenance)

- **Session Colab temporaire** : l'URL change à chaque relance du notebook
- **Latence** : quelques secondes par image sur T4 (chargement + génération)
- **Confiance heuristique** (0.70 / 0.50), non calibrée — limite documentée du modèle `-pt`
- Le warning français obligatoire est réappliqué par `apply_safety_guardrails` du repo

> /!\\ Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.